In [2]:
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration, T5Tokenizer, T5ForConditionalGeneration
import torch

d:\Anaconda\envs\POC_STT\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
with open("test.txt", "r", encoding="utf-8") as f:
    input_text = f.read()

print(input_text)

In [3]:
kobart_tokenizer = PreTrainedTokenizerFast.from_pretrained('digit82/kobart-summarization')
kobart_model = BartForConditionalGeneration.from_pretrained('digit82/kobart-summarization').to("cuda" if torch.cuda.is_available() else "cpu")

d:\Anaconda\envs\POC_STT\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diabl\.cache\huggingface\hub\models--digit82--kobart-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The numbe

In [4]:
t5_tokenizer = T5Tokenizer.from_pretrained("KETI-AIR/ke-t5-base")
t5_model = T5ForConditionalGeneration.from_pretrained("KETI-AIR/ke-t5-base").to("cuda" if torch.cuda.is_available() else "cpu")

d:\Anaconda\envs\POC_STT\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diabl\.cache\huggingface\hub\models--KETI-AIR--ke-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expe

In [5]:
def summarize_kobart(text):
    inputs = kobart_tokenizer.encode(text, return_tensors="pt", max_length=1024, truncation=True)
    inputs = inputs.to(kobart_model.device)
    summary_ids = kobart_model.generate(inputs, max_length=256, num_beams=4, early_stopping=True)
    summary = kobart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary.strip()

In [6]:
def generate_story(summary_text):
    prompt = f"다음 내용을 바탕으로 어린이를 위한 동화 줄거리 5페이지 분량으로 바꿔주세요:\n{summary_text}"
    inputs = t5_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    inputs = inputs.to(t5_model.device)
    output = t5_model.generate(inputs.input_ids, max_new_tokens=300, num_beams=4)
    story = t5_tokenizer.decode(output[0], skip_special_tokens=True)
    return story.strip()

In [ ]:
summary = summarize_kobart(input_text)
story = generate_story(summary)

In [ ]:
print("\n📌 요약 결과:\n", summary)
print("\n📚 동화 줄거리:\n", story)